# Verify Module 4: LRLP Semantic Encoding

The simplest module -- a plain `nn.Embedding(38, embed_dim)` keyed by
landmark index, no image or coordinate data involved, trivial to test in
isolation before wiring into the rest. This notebook does exactly that:
no real GRID/LRS3 data is needed.

**What "looks right" means:**
- Output shape `(B, 38, embed_dim, T)`, identical across every batch
  element and every frame (the embedding encodes landmark identity only).
- The 38 landmarks' embeddings are pairwise distinct (a degenerate,
  collapsed embedding table would defeat the module's purpose).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch

from fusion_avsr.models.landmark.lrlp import NUM_LRLPS
from fusion_avsr.models.landmark.semantic_embedding import DEFAULT_EMBED_DIM, LRLPSemanticEmbedding

module = LRLPSemanticEmbedding(embed_dim=DEFAULT_EMBED_DIM)
print(module)

In [ ]:
B, T = 2, 5
output = module(batch_size=B, num_frames=T)
print("output shape:", tuple(output.shape))
assert output.shape == (B, NUM_LRLPS, DEFAULT_EMBED_DIM, T)

# Identical across batch and time.
assert torch.allclose(output[0], output[1])
assert torch.allclose(output[:, :, :, 0], output[:, :, :, -1])
print("OK: identical across batch and time")

In [ ]:
per_landmark = output[0, :, :, 0]  # (K, D)
pairwise_equal = [
    torch.allclose(per_landmark[i], per_landmark[j])
    for i in range(NUM_LRLPS) for j in range(i + 1, NUM_LRLPS)
]
print(f"{sum(pairwise_equal)}/{len(pairwise_equal)} landmark pairs have identical embeddings (should be 0)")
assert not any(pairwise_equal)

## Checklist

- [ ] `output.shape == (2, 38, 512, 5)`.
- [ ] Identical across batch and time (asserted above).
- [ ] No two landmarks share the same embedding (asserted above).